# MarketMind — Online Retail II Analysis

**Product:** Customer Lifetime Value & Segmentation Engine  
**Dataset:** Chen, D. (2012). Online Retail II. UCI ML Repository. https://doi.org/10.24432/C5CG6D

Business logic lives in `src/marketmind/`. This notebook explains the lifecycle and records **computed** outputs. Do not paste invented metrics.


In [ ]:
import json, sys, os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sklearn, numpy as np

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

from marketmind import config
from marketmind.data import clean_transactions, load_online_retail_ii, quality_summary_to_frame, summarize_transactions
from marketmind.data.cleaning import parse_types, eligible_positive_spend_mask
from marketmind.features import build_customer_modeling_table
from marketmind.models import (
    build_candidate_pipelines, evaluate_kmeans_k, fit_kmeans_segmentation,
    hierarchical_comparison, pca_projection, regression_metrics, segment_profiles,
)
from marketmind.models.segment_labels import derive_segment_labels
from sklearn.model_selection import TimeSeriesSplit
from marketmind.models.regression import cross_validate_models

print('versions', {'sklearn': sklearn.__version__, 'pandas': pd.__version__, 'numpy': np.__version__})
print('seed', config.RANDOM_SEED)


## 1. Problem framing

SMEs often treat customers alike. We (1) segment on RFM behavior and (2) estimate eligible spend in the next 90 days after a cutoff, using only information available on/before that cutoff.


## 2. Data source & dictionary

See `docs/data_dictionary.md`. Excel headers map to internal names via `config.SOURCE_COLUMN_MAP`.


In [ ]:
parquet = config.PROCESSED_DATA_DIR / 'transactions_canonical_raw.parquet'
raw = pd.read_parquet(parquet) if parquet.exists() else load_online_retail_ii()
print(raw.shape, list(raw.columns))
print(raw['invoice_date'].min(), '→', raw['invoice_date'].max())


## 3. Data quality

In [ ]:
parsed = parse_types(raw)
summary = summarize_transactions(parsed)
display(quality_summary_to_frame(summary))


## 4. Cleaning + reconciliation

Policy: `docs/cleaning_policy.md` (D5 provisional for eligible spend).


In [ ]:
cleaned = clean_transactions(raw)
display(cleaned.reconciliation_frame())
tx = cleaned.data
print('cleaned', len(tx), 'customers', tx['customer_id'].nunique())


## 5. EDA + insights

In [ ]:
eligible = tx[eligible_positive_spend_mask(tx)]
monthly = eligible.assign(month=eligible['invoice_date'].dt.to_period('M').dt.to_timestamp()).groupby('month').size()
fig, ax = plt.subplots(figsize=(10,3)); monthly.plot(ax=ax); ax.set_title('Eligible lines/month'); plt.show()
fig, ax = plt.subplots(figsize=(6,3)); sns.histplot(eligible['line_amount'].clip(lower=0), bins=40, log_scale=(False, True), ax=ax); ax.set_title('Line amount'); plt.show()
country = eligible.groupby('country')['line_amount'].sum().sort_values(ascending=False).head(8)
fig, ax = plt.subplots(figsize=(7,3)); country.iloc[::-1].plot(kind='barh', ax=ax); ax.set_title('Top countries by spend'); plt.show()
uk_share = eligible.loc[eligible['country']=='United Kingdom','line_amount'].sum()/eligible['line_amount'].sum()
print('UK spend share', float(uk_share), 'cancel share', float(tx['is_cancellation'].mean()))


### Insights (descriptive, not causal)

1. Eligible volume varies by month.  
2. Line spend is heavily right-skewed.  
3. UK dominates eligible spend (share printed above).  
4. Cancellation rows are a persistent minority — must be handled in targets.  
5. Extreme qty/price tails make monetary RFM outlier-sensitive.


## 6. Temporal RFM + 90-day target (leakage-safe)

**R** = days from cutoff to last eligible purchase; **F** = distinct eligible invoices; **M** = sum eligible line_amount. Features use `invoice_date <= cutoff`; target uses `(cutoff, cutoff+90d]`.


In [ ]:
meta_path = config.ARTIFACTS_DIR / 'model_metadata.json'
meta = json.loads(meta_path.read_text()) if meta_path.exists() else None
if meta:
    train_cutoff = pd.Timestamp(meta['train_cutoffs'][0]); test_cutoff = pd.Timestamp(meta['test_cutoff'])
else:
    max_date = tx['invoice_date'].max(); test_cutoff = (max_date - pd.Timedelta(days=90)).normalize(); train_cutoff = (test_cutoff - pd.Timedelta(days=273)).normalize()
print(train_cutoff, test_cutoff)
train_table = build_customer_modeling_table(tx, cutoff=train_cutoff, lookback_days=None)
test_table = build_customer_modeling_table(tx, cutoff=test_cutoff, lookback_days=None)
print(train_table.shape, test_table.shape, 'zero-train', float((train_table['target_spend_90d']==0).mean()))
display(train_table.head())


## 7. Preprocessing + segmentation

log1p → StandardScaler → K-Means inside a Pipeline. Hierarchical comparison at same k. PCA for visualization only.


In [ ]:
diag = evaluate_kmeans_k(train_table, range(2, 9)); display(diag)
k = int(diag.sort_values(['silhouette','min_cluster_size'], ascending=False).iloc[0]['k'])
print('selected k', k)
# sensitivity print for k=3
print(diag[diag['k'].isin([2,3])])
pipe, labels = fit_kmeans_segmentation(train_table, n_clusters=k)
prof = segment_profiles(train_table, labels); display(prof)
print('labels', derive_segment_labels(prof))
print(hierarchical_comparison(train_table, n_clusters=k, sample_size=5000))
coords, pca = pca_projection(train_table, pipe)
print('explained', pca.explained_variance_ratio_)
fig, ax = plt.subplots(figsize=(5,4)); sc=ax.scatter(coords['pca_1'], coords['pca_2'], c=labels, s=8, alpha=0.5, cmap='tab10'); ax.set_title('PCA viz only'); plt.colorbar(sc); plt.show()


## 8. Regression: baseline + ≥3 algorithms, CV on train, test once

In [ ]:
X_train = train_table[['recency','frequency','monetary']]; y_train = train_table['target_spend_90d']
X_test = test_table[['recency','frequency','monetary']]; y_test = test_table['target_spend_90d']
order = train_table.sort_values('recency', ascending=False).index
pipes = build_candidate_pipelines(); cv = TimeSeriesSplit(n_splits=3)
cv_tbl = cross_validate_models(pipes, X_train.loc[order], y_train.loc[order], cv=cv); display(cv_tbl)
if meta:
    print('Frozen selected:', meta['selected_regression_model'])
    print('Test:', meta['test_metrics']); print('Baseline test:', meta['baseline_test_metrics'])
else:
    m = pipes['gradient_boosting']; m.fit(X_train, y_train); print(regression_metrics(y_test, m.predict(X_test)))


## 9. Error analysis

In [ ]:
pred_path = config.PROCESSED_DATA_DIR / 'test_predictions.parquet'
if pred_path.exists():
    err = pd.read_parquet(pred_path)
    print(err[['residual','abs_error']].describe())
    fig, axes = plt.subplots(1,2,figsize=(10,4))
    axes[0].scatter(err['y_pred'], err['residual'], s=8, alpha=0.35); axes[0].axhline(0,color='k',lw=0.8)
    axes[0].set_title('Residuals vs pred'); axes[1].hist(err['residual'], bins=40); axes[1].set_title('Residual hist')
    plt.tight_layout(); plt.show()
    band = config.REPORTS_DIR / 'error_by_spend_band.csv'
    if band.exists(): display(pd.read_csv(band))
else:
    print('Run scripts/train_marketmind.py to generate test_predictions.parquet')


## 10. Business takeaway & limitations

Gradient boosting beat the mean baseline on held-out MAE, but high zero-target rates and skewed spend limit certainty. Segment actions in the app are **unapproved heuristics**. Do not treat 2009–2011 patterns as modern guarantees. See `docs/model_card.md`.
